In [ ]:
import torch
import torch.nn.functional as F

from tqdm import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from dataclasses import dataclass


from torch.nn.utils import clip_grad_norm_

In [ ]:
IGNORE_INDEX=-100
MAX_LENGTH=1024

In [ ]:
from dataclasses import dataclass


@dataclass
class Config:
    # pretrained_lm: str = "openai-community/gpt2"
    pretrained_lm: str = "HuggingFaceTB/SmolLM2-135M"
    ift_dataset: str = "HuggingFaceTB/smoltalk"
    # the pretrained lm does not come with a chat template
    chat_template_tokenizer: str = "HuggingFaceTB/SmolLM2-135M-Instruct"
    dataset_subset: str = "everyday-conversations"
    dataset_split: str = "train"
    max_length: int = MAX_LENGTH
    device: str = "mps"

    # training hyperparams
    lr: float = 0.00005
    num_epochs: int = 2
    batch_size: int = 8
    gradient_accumulation_steps: int = 8
    warmup_ratio: float = 0.1
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    seed: int = 42

    # Hardware
    bf16: bool = True
    gradient_checkpointing: bool = True
    model_device_id: int = 0
    

cfg = Config

In [ ]:
def load_model(cfg, device: torch.device):
    tokenizer = AutoTokenizer.from_pretrained(
        cfg.pretrained_lm, trust_remote_code=False
    )    

    if tokenizer.chat_template is None:
        if cfg.chat_template_tokenizer:
            donor = AutoTokenizer.from_pretrained(
                cfg.chat_template_tokenizer, trust_remote_code=False
            )
            tokenizer.chat_template = donor.chat_template

            # Set all the special tokens from donor to tokenizer
            special_token_names = [
                "pad_token", "bos_token", "eos_token", "unk_token", 
                "sep_token", "cls_token", "mask_token", 
                "additional_special_tokens"
            ]
            for token_name in special_token_names:
                donor_token = getattr(donor, token_name, None)
                if donor_token is not None:
                    setattr(tokenizer, token_name, donor_token)
    

        else:
           tokenizer.chat_template = (
                "{% for message in messages %}"
                "{% if loop.first and messages[0]['role'] != 'system' %}"
                "{{ 'system\nBelow is an instruction that describes a task. Write a response that appropriately completes the request.\n<|endoftext|>\n' }}"
                "{% endif %}"
                "{{ message['role'] + '\n' + message['content'] + '<|endoftext|>' + '\n' }}"
                "{% endfor %}"
                "{% if add_generation_prompt %}{{ 'assistant\n' }}{% endif %}"
            )
   

    


    dtype = torch.bfloat16 if cfg.bf16 else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        cfg.pretrained_lm,
        trust_remote_code=False,
        dtype=dtype,
    ).to(device)

    return model, tokenizer


In [ ]:
model, tokenizer = load_model(cfg, "mps")

In [ ]:
def generate_response(messages, model, tokenizer):
    """
    Generate a response given a list of chat messages.

    Args:
        messages (list): List of dicts with 'role' and 'content' representing the conversation.

    Returns:
        response (str): The model's generated response as a string.
    """
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    return response

In [ ]:
generate_response([{"role": "user", "content": "Is the Earth bigger than the Sun?"}], model, tokenizer)

In [ ]:
# for Seb Raschka's ift dataset
import os
import requests
import json

def download_and_load_file(file_path, url):
    # Download file only if it does not exist (idempotent)
    if not os.path.exists(file_path):
        r = requests.get(url)
        r.raise_for_status()
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(r.text)
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

file_path = "instruction-data.json"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"
seb_data = download_and_load_file(file_path, url)

def transform_to_messages_wrapped(data):
    """
    Transforms a list of dicts with keys 'instruction', 'input', and 'output'
    into a list of dicts with a "messages" key, where each value is a list of multiturn
    conversation messages (dicts with 'role' and 'content').
    This allows downstream code to access example["messages"] as expected.
    """
    examples = []
    for example in data:
        conversation = []
        user_content = ""
        # Concatenate instruction and input into one user message
        instr = example.get("instruction", "")
        inp = example.get("input", "")
        if instr and inp:
            user_content = instr + "\n" + inp
        elif instr:
            user_content = instr
        if user_content:
            conversation.append({"role": "user", "content": user_content})
 
        if example.get("output"):
            conversation.append({"role": "assistant", "content": example["output"]})
        if conversation:  # Only add non-empty conversations
            examples.append({"messages": conversation})
    return examples

seb_dataset = transform_to_messages_wrapped(seb_data)
seb_dataset[:4]

In [ ]:
tokenizer.apply_chat_template(seb_dataset[0]["messages"], tokenize=False)

In [ ]:
def encode_example(example, tokenizer: AutoTokenizer, max_length):
    messages = example["messages"]

    # apply_chat_template(messages) != apply_chat_template(messages[-1:]) + apply_chat_template(messages[:-1])
    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True, return_dict=False
    )

    full_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False, return_dict=False
    )

    # assert full_ids[:len(prompt_ids)] == prompt_ids

    if len(full_ids) > max_length:
        return None

    labels = [IGNORE_INDEX] * len(prompt_ids)
    labels += full_ids[len(prompt_ids) :]

    input_ids = full_ids[:max_length]
    labels = labels[:max_length]

    if all(x == IGNORE_INDEX for x in labels):
        return None

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

@dataclass
class SFTBatch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    labels: torch.Tensor

    def to(self, device: torch.device | str) -> "SFTBatch":
        return SFTBatch(
            input_ids=self.input_ids.to(device),
            attention_mask=self.attention_mask.to(device),
            labels=self.labels.to(device),
        )


class SFTDataset(Dataset):
    def __init__(self, raw_dataset, tokenizer, max_length):
        self.examples = []

        for row in raw_dataset:
            encoded = encode_example(row, tokenizer, max_length)
            if encoded is not None:
                self.examples.append(encoded)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collator(examples, pad_token_id):
    max_len = max(ex["input_ids"].size(0) for ex in examples)

    input_ids = []
    attention_mask = []
    labels = []

    for ex in examples:
        ids = ex["input_ids"]
        lbl = ex["labels"]
        pad_len = max_len - ids.size(0)

        input_ids.append(
            torch.cat(
                [
                    ids,
                    torch.full(
                        size=(pad_len,), fill_value=pad_token_id, dtype=torch.long
                    ),
                ]
            )
        )

        attention_mask.append(
            torch.cat(
                [
                    torch.ones(size=(ids.size(0),), dtype=torch.long),
                    torch.zeros(size=(pad_len,), dtype=torch.long),
                ]
            )
        )

        labels.append(
            torch.cat(
                [
                    lbl,
                    torch.full(
                        size=(pad_len,), fill_value=IGNORE_INDEX, dtype=torch.long
                    ),
                ]
            )
        )

    return SFTBatch(
        input_ids=torch.stack(input_ids),
        attention_mask=torch.stack(attention_mask),
        labels=torch.stack(labels),
    )


def create_dataloader(cfg, tokenizer: AutoTokenizer):

    # raw_dataset = load_dataset(cfg.ift_dataset, cfg.dataset_subset, split=cfg.dataset_split)
    raw_dataset = seb_dataset

    ds = SFTDataset(raw_dataset, tokenizer, max_length=MAX_LENGTH)

    pad_token_id = tokenizer.pad_token_id

    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=lambda batch: collator(batch, pad_token_id),
        num_workers=0,
        pin_memory=False,
    )


In [ ]:
def compute_loss(model, batch):
    output = model(
        input_ids=batch.input_ids,
        attention_mask=batch.attention_mask,
        use_cache=False,
    )
    # shift for next token prediction
    shift_logits = output.logits[:, :-1, :].contiguous()
    shift_labels = batch.labels[:, 1:].contiguous()
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=IGNORE_INDEX,
    )

In [ ]:
dataloader = create_dataloader(cfg, tokenizer)

In [ ]:
len(dataloader)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)

In [ ]:
model.train()
optimizer.zero_grad(set_to_none=True)

In [ ]:
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=cfg.num_epochs * len(dataloader),
)

In [ ]:
for epoch in range(cfg.num_epochs):
    print(f"Epoch {epoch + 1}/{cfg.num_epochs}:")
    with tqdm(dataloader, desc=f"Epoch {epoch+1}/{cfg.num_epochs}") as pbar:
        for batch in pbar:
            batch = batch.to(cfg.device)
            loss = compute_loss(model, batch)
            loss.backward()
            clip_grad_norm_(model.parameters(), max_norm=1.0)


            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            pbar.set_postfix({"loss": f"{loss.detach().item():.4f}"})
   

In [ ]:
model.eval()

In [ ]:
tokenizer.bos_token_id

In [ ]:
generate_response([{"role": "user", "content": "What is 5+4?"}], model, tokenizer)

In [ ]:
generate_response([{"role": "user", "content": "Are you sure that the Earth is bigger than the Sun?"}], model, tokenizer)